# Pneumonia Detection and Type Classification from Chest X-Rays (Classical Machine Learning)

Each chest X-ray is classified into one of three classes:
- 0: Normal
- 1: Bacterial pneumonia
- 2: Viral pneumonia

No neural network is used. Features are extracted by hand from every image (shape, texture and brightness information), and five classical machine learning models are trained on those numbers:
Logistic Regression, K-Nearest Neighbors, Support Vector Machine, Random Forest and XGBoost.

Everything runs on a normal CPU.

## 1. Libraries and setup

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import joblib
from joblib import Parallel, delayed
from scipy.stats import skew, kurtosis
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             accuracy_score, precision_score, recall_score, f1_score)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 128
CLASSES = ["Normal", "Bacterial pneumonia", "Viral pneumonia"]
SHORT = ["Normal", "Bacterial", "Viral"]
NUM_CLASSES = len(CLASSES)

## 2. Dataset location

Put the path of your `dataset` folder in `DATA_PATH`. If this notebook sits right next to the `dataset` folder, `"dataset"` already works.
On Windows keep the `r` before the quotes, for example `r"C:\Users\YourName\Downloads\dataset"`.

Expected layout:
```
dataset/
├── labels_train.csv
├── train_images/train_images/   (labeled images)
└── test_images/test_images/     (images without labels)
```
The code finds the CSV and both image folders by itself, even if they are nested differently.

In [ ]:
DATA_PATH = r"dataset"

IMAGE_TYPES = (".jpg", ".jpeg", ".png")

def find_labels_csv(base):
    for root, dirs, files in os.walk(base):
        for name in files:
            if name.lower().startswith("labels_train") and name.lower().endswith(".csv"):
                return os.path.join(root, name)
    raise FileNotFoundError("labels_train.csv not found inside " + os.path.abspath(base))

def find_image_folder(base, folder_name):
    for root, dirs, files in os.walk(base):
        if os.path.basename(root) == folder_name and any(f.lower().endswith(IMAGE_TYPES) for f in files):
            return root
    return None

LABELS_CSV = find_labels_csv(DATA_PATH)
TRAIN_DIR = find_image_folder(DATA_PATH, "train_images")
TEST_DIR = find_image_folder(DATA_PATH, "test_images")

print("Labels file:   ", LABELS_CSV)
print("Train images:  ", TRAIN_DIR)
print("Test images:   ", TEST_DIR)

## 3. Reading the labels

The CSV has two columns: `file_name` and `class_id` (0 = Normal, 1 = Bacterial pneumonia, 2 = Viral pneumonia).
Each file name is joined with the train image folder. Any image listed in the CSV but missing on disk is skipped, so deleting some images to save space is fine.

In [ ]:
labels = pd.read_csv(LABELS_CSV)
labels.columns = labels.columns.str.strip()
labels["path"] = labels["file_name"].map(lambda name: os.path.join(TRAIN_DIR, name))
on_disk = labels["path"].map(os.path.exists)

df = labels[on_disk].rename(columns={"class_id": "label"}).reset_index(drop=True)
print("Images in CSV:", len(labels), "| found on disk:", len(df), "| missing:", (~on_disk).sum())
df["label"].map(dict(enumerate(CLASSES))).value_counts()

## 4. Train and test split

The images in `test_images` have no labels, so they cannot be used to measure accuracy.
Instead, 20% of the labeled data is held out as the test set (stratified, so every class keeps its proportion).
A separate validation set is not needed, because model tuning uses 5-fold cross-validation on the training part.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=SEED)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
y_train = train_df["label"].values
y_test = test_df["label"].values

summary = pd.DataFrame({name: frame["label"].value_counts().sort_index()
                        for name, frame in [("Train", train_df), ("Test", test_df)]})
summary.index = CLASSES
summary.loc["Total"] = summary.sum()
summary

## 5. Preprocessing

Each X-ray is loaded in grayscale, resized to 128 x 128 and passed through CLAHE (Contrast Limited Adaptive Histogram Equalization).
CLAHE improves local contrast, which makes the cloudy pneumonia patches stand out more clearly from healthy lung tissue.

In [ ]:
def read_xray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)

sample_paths = [train_df[train_df["label"] == k]["path"].iloc[0] for k in range(NUM_CLASSES)]
plt.figure(figsize=(9, 3 * NUM_CLASSES))
for i, path in enumerate(sample_paths):
    original = cv2.resize(cv2.imread(path, cv2.IMREAD_GRAYSCALE), (IMG_SIZE, IMG_SIZE))
    plt.subplot(NUM_CLASSES, 2, i * 2 + 1)
    plt.imshow(original, cmap="gray")
    plt.title(f"{CLASSES[i]} - original")
    plt.axis("off")
    plt.subplot(NUM_CLASSES, 2, i * 2 + 2)
    plt.imshow(read_xray(path), cmap="gray")
    plt.title(f"{CLASSES[i]} - after CLAHE")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Feature extraction

Every image is turned into one long vector of numbers made of four groups:

- HOG (Histogram of Oriented Gradients): direction and strength of edges in small cells, which describes shapes like the lung borders, ribs and heart.
- LBP (Local Binary Patterns): compares each pixel with its neighbours to describe texture, at two scales.
- GLCM (Gray Level Co-occurrence Matrix): how often pairs of gray levels appear next to each other, giving texture measures such as contrast, homogeneity and energy.
- Intensity statistics: mean, standard deviation, skewness, kurtosis, percentiles and a 32-bin brightness histogram.

Bacterial pneumonia usually appears as one dense white patch in a lobe, while viral pneumonia is more spread out and patchy across both lungs. Texture features are what give the models a chance to tell them apart.

In [ ]:
GLCM_PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

def feature_parts(img):
    hog_feat = hog(img, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")

    lbp_feat = []
    for points, radius in [(8, 1), (16, 2)]:
        lbp = local_binary_pattern(img, points, radius, method="uniform")
        hist, _ = np.histogram(lbp, bins=points + 2, range=(0, points + 2), density=True)
        lbp_feat.append(hist)

    quantized = (img // 4).astype(np.uint8)
    glcm = graycomatrix(quantized, distances=[1, 3], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                        levels=64, symmetric=True, normed=True)
    glcm_feat = np.concatenate([graycoprops(glcm, prop).mean(axis=1) for prop in GLCM_PROPS])

    pixels = img.ravel().astype(np.float32)
    stats = [pixels.mean(), pixels.std(), skew(pixels), kurtosis(pixels)]
    stats += list(np.percentile(pixels, [10, 25, 50, 75, 90]))
    hist, _ = np.histogram(pixels, bins=32, range=(0, 256), density=True)

    return {
        "HOG": hog_feat,
        "LBP": np.concatenate(lbp_feat),
        "GLCM": glcm_feat,
        "Intensity": np.concatenate([stats, hist]),
    }

def extract_features(path):
    parts = feature_parts(read_xray(path))
    return np.concatenate(list(parts.values())).astype(np.float32)

example = feature_parts(read_xray(sample_paths[0]))
feature_groups = np.concatenate([[name] * len(values) for name, values in example.items()])
for name, values in example.items():
    print(f"{name:<10} {len(values):>5} features")
print(f"{'Total':<10} {len(feature_groups):>5} features per image")

### What HOG and LBP actually see

In [ ]:
plt.figure(figsize=(12, 4 * NUM_CLASSES))
for i, path in enumerate(sample_paths):
    img = read_xray(path)
    _, hog_image = hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    lbp_image = local_binary_pattern(img, 8, 1, method="uniform")
    for j, (title, picture) in enumerate([("X-ray", img), ("HOG", hog_image), ("LBP", lbp_image)]):
        plt.subplot(NUM_CLASSES, 3, i * 3 + j + 1)
        plt.imshow(picture, cmap="gray" if j != 1 else "inferno")
        plt.title(f"{SHORT[i]} - {title}")
        plt.axis("off")
plt.tight_layout()
plt.show()

### Extracting features for all images

This runs on all CPU cores in parallel. The result is saved to a file, so if the notebook restarts the features are loaded back instead of computed again.
Delete `xray_features.npz` if you change the dataset or the feature code.

In [ ]:
def build_matrix(paths):
    features = Parallel(n_jobs=-1, verbose=5)(delayed(extract_features)(p) for p in paths)
    return np.vstack(features)

FEATURE_FILE = "xray_features.npz"

if os.path.exists(FEATURE_FILE):
    saved = np.load(FEATURE_FILE)
    X_train, X_test = saved["X_train"], saved["X_test"]
else:
    start = time.time()
    X_train = build_matrix(train_df["path"])
    X_test = build_matrix(test_df["path"])
    np.savez_compressed(FEATURE_FILE, X_train=X_train, X_test=X_test)
    print(f"Done in {time.time() - start:.0f} seconds")

print("Train matrix:", X_train.shape)
print("Test matrix: ", X_test.shape)

## 7. Models and hyperparameter tuning

- Logistic Regression, KNN and SVM work with distances or weighted sums, so features are standardized first and PCA keeps the components that explain 95% of the variance.
- Random Forest and XGBoost are tree-based, so they get the raw features.

Class imbalance is handled with `class_weight="balanced"`, and with balanced sample weights for XGBoost.

Each model is tuned with GridSearchCV using 5-fold stratified cross-validation. Macro F1 is the score used to pick the best settings, because it treats all three classes equally.

In [ ]:
def scaled_pca(classifier):
    return Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=SEED)),
        ("clf", classifier),
    ])

xgb_weights = {"clf__sample_weight": compute_sample_weight("balanced", y_train)}

models = {
    "Logistic Regression": (
        scaled_pca(LogisticRegression(class_weight="balanced", max_iter=5000)),
        {"clf__C": [0.01, 0.1, 1]}, {},
    ),
    "KNN": (
        scaled_pca(KNeighborsClassifier(weights="distance")),
        {"clf__n_neighbors": [5, 9, 15]}, {},
    ),
    "SVM": (
        scaled_pca(SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=SEED)),
        {"clf__C": [1, 10], "clf__gamma": ["scale", 0.001]}, {},
    ),
    "Random Forest": (
        Pipeline([("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=1))]),
        {"clf__max_depth": [None, 20], "clf__min_samples_leaf": [1, 3]}, {},
    ),
    "XGBoost": (
        Pipeline([("clf", XGBClassifier(n_estimators=300, subsample=0.8, colsample_bytree=0.5,
                                        tree_method="hist", eval_metric="mlogloss", random_state=SEED, n_jobs=1))]),
        {"clf__max_depth": [4, 6], "clf__learning_rate": [0.05, 0.1]}, xgb_weights,
    ),
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fitted = {}
cv_rows = []

for name, (pipeline, grid, fit_params) in models.items():
    start = time.time()
    search = GridSearchCV(pipeline, grid, scoring="f1_macro", cv=cv, n_jobs=-1)
    search.fit(X_train, y_train, **fit_params)
    fitted[name] = search.best_estimator_
    params = {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
    cv_rows.append({"Model": name, "CV macro F1": round(search.best_score_, 4),
                    "Best settings": params, "Time (s)": round(time.time() - start, 1)})
    print(f"{name:<20} CV macro F1 {search.best_score_:.4f}  {params}  ({time.time() - start:.0f}s)")

cv_results = pd.DataFrame(cv_rows).set_index("Model")
cv_results

## 8. Evaluation on the test set

Besides the 3-class metrics (macro averages treat all three classes equally), two extra numbers answer the most practical question, "is there pneumonia at all?", by merging bacterial and viral into one pneumonia group:
- Pneumonia detection recall: how many pneumonia patients were flagged as having some pneumonia
- Pneumonia detection specificity: how many healthy people were correctly called normal

In [ ]:
test_rows = []
test_probs = {}
fig, axes = plt.subplots(1, len(fitted), figsize=(4.4 * len(fitted), 4.2))

for ax, (name, model) in zip(axes, fitted.items()):
    y_pred = model.predict(X_test)
    probs = model.predict_proba(X_test)
    test_probs[name] = probs
    sick_true = (y_test > 0).astype(int)
    sick_pred = (y_pred > 0).astype(int)
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro precision": precision_score(y_test, y_pred, average="macro"),
        "Macro recall": recall_score(y_test, y_pred, average="macro"),
        "Macro F1": f1_score(y_test, y_pred, average="macro"),
        "ROC AUC (one-vs-rest)": roc_auc_score(y_test, probs, multi_class="ovr"),
        "Pneumonia detection recall": recall_score(sick_true, sick_pred),
        "Pneumonia detection specificity": recall_score(sick_true, sick_pred, pos_label=0),
    })
    cm = confusion_matrix(y_test, y_pred, labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=SHORT, yticklabels=SHORT, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

results = pd.DataFrame(test_rows).set_index("Model").round(4)
results.T

In [ ]:
results[["Accuracy", "Macro F1", "ROC AUC (one-vs-rest)", "Pneumonia detection recall"]].plot(
    kind="bar", figsize=(12, 5), width=0.8)
plt.title("Test set comparison of classical ML models")
plt.ylabel("Score")
plt.ylim(0.3, 1.0)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 9. Best model

The best model is chosen by cross-validation macro F1, not by test score, so the test set is never used to make a decision.

In [ ]:
best_name = cv_results["CV macro F1"].idxmax()
best_model = fitted[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best_model.predict(X_test), target_names=CLASSES, digits=4))

## 10. Which features mattered most?

Random Forest gives an importance score to every feature. Adding these up per group shows which kind of information the model relied on.

In [ ]:
forest = fitted["Random Forest"].named_steps["clf"]
importance = pd.Series(forest.feature_importances_).groupby(feature_groups).sum().sort_values()
counts = pd.Series(feature_groups).value_counts()

importance.plot(kind="barh", figsize=(7, 3.5), color="#4C72B0")
plt.title("Total Random Forest importance per feature group")
plt.xlabel("Importance")
plt.show()

pd.DataFrame({"Features": counts, "Total importance": importance.round(4),
              "Importance per feature": (importance / counts).round(6)}).sort_values("Total importance", ascending=False)

## 11. Saving the model and predicting a new X-ray

In [ ]:
joblib.dump(best_model, "pneumonia_classical_ml.joblib")

def predict_xray(image_path, model=best_model):
    features = extract_features(image_path).reshape(1, -1)
    probs = model.predict_proba(features)[0]
    verdict = CLASSES[int(probs.argmax())]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(read_xray(image_path), cmap="gray")
    ax1.set_title(f"{best_name}: {verdict}")
    ax1.axis("off")
    ax2.barh(SHORT, probs, color=["#4C72B0", "#C44E52", "#DD8452"])
    ax2.set_xlim(0, 1)
    ax2.set_title("Predicted probabilities")
    plt.tight_layout()
    plt.show()
    return verdict, dict(zip(CLASSES, probs.round(4)))

predict_xray(test_df["path"].iloc[0])

## 12. Predicting the unlabeled test images

The images in `test_images` have no labels, so they cannot be scored. The best model predicts a class for each one and the results are saved to `test_predictions_ml.csv`.

In [ ]:
test_files = sorted(f for f in os.listdir(TEST_DIR) if f.lower().endswith(IMAGE_TYPES)) if TEST_DIR else []
print("Unlabeled test images:", len(test_files))

if test_files:
    X_unlabeled = build_matrix([os.path.join(TEST_DIR, f) for f in test_files])
    predicted = best_model.predict(X_unlabeled)
    output = pd.DataFrame({"file_name": test_files, "class_id": predicted})
    output.to_csv("test_predictions_ml.csv", index=False)
    print(output["class_id"].map(dict(enumerate(CLASSES))).value_counts())
    print("Saved to test_predictions_ml.csv")